In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark=SparkSession.builder.appName("CourseAnalysis").getOrCreate()

print("Spark is created")

Spark is created


In [2]:
from google.colab  import files
uploaded=files.upload()

Saving enrollments.csv to enrollments.csv


In [3]:
from google.colab  import files
uploaded=files.upload()

Saving progress.csv to progress.csv


In [5]:
enroll_df=spark.read.csv("enrollments.csv",header=True,inferSchema=True)
progress_df=spark.read.csv("progress.csv",header=True,inferSchema=True)
enroll_df.show()
progress_df.show()

+-------------+----------+---------+---------------+---------+
|enrollment_id|student_id|course_id|enrollment_date|   status|
+-------------+----------+---------+---------------+---------+
|            1|         1|      101|     2025-02-01|   Active|
|            2|         2|      102|     2025-02-02|Completed|
|            3|         3|      103|     2025-02-03|   Active|
|            4|         4|      104|     2025-02-04|  Dropped|
|            5|         5|      105|     2025-02-05|Completed|
|            6|         6|      106|     2025-02-06|   Active|
|            7|         7|      107|     2025-02-07|Completed|
|            8|         8|      108|     2025-02-08|  Dropped|
|            9|         9|      109|     2025-02-09|   Active|
|           10|        10|      110|     2025-02-10|Completed|
+-------------+----------+---------+---------------+---------+

+----------+---------+----------+
|student_id|course_id|completion|
+----------+---------+----------+
|         1|   

In [11]:
progress_df = progress_df.withColumnRenamed(
    "course_id",
    "progress_course_id"
)

In [12]:
joined=enroll_df.join(progress_df,"student_id")
joined.show()

+----------+-------------+---------+---------------+---------+------------------+----------+
|student_id|enrollment_id|course_id|enrollment_date|   status|progress_course_id|completion|
+----------+-------------+---------+---------------+---------+------------------+----------+
|         1|            1|      101|     2025-02-01|   Active|               101|        45|
|         2|            2|      102|     2025-02-02|Completed|               102|       100|
|         3|            3|      103|     2025-02-03|   Active|               103|        65|
|         4|            4|      104|     2025-02-04|  Dropped|               104|        20|
|         5|            5|      105|     2025-02-05|Completed|               105|       100|
|         6|            6|      106|     2025-02-06|   Active|               106|        55|
|         7|            7|      107|     2025-02-07|Completed|               107|       100|
|         8|            8|      108|     2025-02-08|  Dropped|        

In [13]:
result = joined.groupBy("course_id").agg(

    count(
        when(col("completion") >= 100, True)
    ).alias("completed_students"),

    count(
        when(col("completion") < 50, True)
    ).alias("dropouts")

)

print("Final Report")
result.show()

Final Report
+---------+------------------+--------+
|course_id|completed_students|dropouts|
+---------+------------------+--------+
|      108|                 0|       1|
|      101|                 0|       1|
|      103|                 0|       0|
|      107|                 1|       0|
|      102|                 1|       0|
|      109|                 0|       1|
|      105|                 1|       0|
|      110|                 1|       0|
|      106|                 0|       0|
|      104|                 0|       1|
+---------+------------------+--------+

